# 02 — Resolve Logical Person Identities

## Responsibility

Load one observation run, cluster face embeddings, detect tracker-ID switches, and save persistent logical person assignments.

This notebook **does not run YOLO, Deep OC-SORT, InsightFace, video decoding, cropping, or rendering**. Change its thresholds and rerun it without repeating observation collection.

## 1. Configuration

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
if not globals().get("PERSON_TRACKER_MASTER", False):
    mode = "ANALYSIS"
mode = str(mode).upper()
if mode not in {"ANALYSIS", "PRODUCTION"}:
    raise ValueError("mode must be 'ANALYSIS' or 'PRODUCTION'")

# Precedence: master-provided RUN_DIRECTORY, explicit override, environment, latest.json.
RUN_DIRECTORY_OVERRIDE = globals().get("RUN_DIRECTORY") if globals().get("PERSON_TRACKER_MASTER", False) else None
if not globals().get("PERSON_TRACKER_MASTER", False):
    RUN_DIRECTORY_OVERRIDE = None  # Set to a run path/name to override runs/latest.json.

    MIN_FACE_QUALITY = 0.50
    CLUSTER_EPS = 0.40
    CLUSTER_MIN_SAMPLES = 4
    MAX_REPRESENTATIVES_PER_TRACK = 12
    ASSIGNMENT_SIMILARITY = 0.60
    MERGE_SIMILARITY = 0.50
    MERGE_TOP_K = 5
    ANCHOR_SMOOTHING_WINDOW = 5
    MIN_SEGMENT_ANCHORS = 2
    MIN_SWITCH_JUMP_SCORE = 0.25
    MAX_FALLBACK_SEARCH_SECONDS = 1.0
    MIN_FALLBACK_OVERLAP = 0.15
    TEMPORAL_DECAY_SECONDS = 1.0
    FALLBACK_CONFIDENCE_SCALE = 0.75
    MIN_FALLBACK_CONFIDENCE = 0.10
    MIN_CONFLICT_CANDIDATE_SCORE = 0.10


## 2. Load the saved observation run

In [ ]:
import sys

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))
get_ipython().run_line_magic("load_ext", "person_tracker.notebook_magics")
print(f"Mode: {mode}")

from person_tracker.storage import load_observation_run, resolve_run_directory

RUN_DIRECTORY = resolve_run_directory(
    PROJECT_ROOT / "runs", stage="observation", explicit=RUN_DIRECTORY_OVERRIDE,
)
print(f"Resolved run directory: {RUN_DIRECTORY}")

manifest, tracking_history, face_samples = load_observation_run(
    RUN_DIRECTORY,
    load_face_crops=True,
)

width = int(manifest["width"])
height = int(manifest["height"])
fps = float(manifest["fps"])

print(f"Source: {manifest['source_video']}")
print(f"Frames: {manifest['start_frame']}:{manifest['end_frame']}")
print(f"Tracked frames: {len(tracking_history)}")
print(f"Face samples: {len(face_samples)}")
print(f"Collection complete: {manifest['settings'].get('complete')}")

## 3. Resolve persistent logical person IDs

In [ ]:
from person_tracker.identity import (
    build_identity_history_from_boundaries,
    build_track_identity_anchors,
    cluster_face_samples,
    detect_switch_boundaries,
    merge_face_clusters,
    resolve_frame_identity_conflicts,
    resolve_unidentified_tracks,
    smooth_identity_anchors,
    split_track_identity_anchors,
)

valid_face_samples, face_assignments = cluster_face_samples(
    face_samples,
    min_quality=MIN_FACE_QUALITY,
    eps=CLUSTER_EPS,
    min_samples=CLUSTER_MIN_SAMPLES,
    max_representatives_per_track=MAX_REPRESENTATIVES_PER_TRACK,
    assignment_similarity=ASSIGNMENT_SIMILARITY,
)
initial_person_count = len(set(face_assignments.values()))

face_assignments = merge_face_clusters(
    valid_face_samples,
    face_assignments,
    similarity_threshold=MERGE_SIMILARITY,
    top_k=MERGE_TOP_K,
)
merged_person_count = len(set(face_assignments.values()))

track_identity_anchors = build_track_identity_anchors(
    valid_face_samples,
    face_assignments,
)
track_identity_anchors = {
    track_id: smooth_identity_anchors(anchors, window=ANCHOR_SMOOTHING_WINDOW)
    for track_id, anchors in track_identity_anchors.items()
}

tracklets = split_track_identity_anchors(
    track_identity_anchors,
    min_segment_anchors=MIN_SEGMENT_ANCHORS,
)
switch_boundaries = detect_switch_boundaries(
    tracklets,
    tracking_history,
    min_jump_score=MIN_SWITCH_JUMP_SCORE,
)

identity_history = build_identity_history_from_boundaries(
    tracking_history,
    tracklets,
    switch_boundaries,
    track_identity_anchors,
    frame_size=(width, height),
)
identity_history = resolve_unidentified_tracks(
    tracking_history,
    identity_history,
    fps,
    max_search_seconds=MAX_FALLBACK_SEARCH_SECONDS,
    min_overlap=MIN_FALLBACK_OVERLAP,
    temporal_decay_seconds=TEMPORAL_DECAY_SECONDS,
    fallback_confidence_scale=FALLBACK_CONFIDENCE_SCALE,
    min_fallback_confidence=MIN_FALLBACK_CONFIDENCE,
)
identity_history = resolve_frame_identity_conflicts(
    tracking_history,
    identity_history,
    min_candidate_score=MIN_CONFLICT_CANDIDATE_SCORE,
)

print(f"Valid face samples: {len(valid_face_samples)}")
print(f"Persons before merge: {initial_person_count}")
print(f"Persons after merge: {merged_person_count}")
print(f"Tracks with direct face identity: {len(track_identity_anchors)}")

## 4. Save identity assignments and events

In [ ]:
from collections import Counter

from person_tracker.storage import save_identity_resolution, update_latest_run

assignment_count = sum(len(items) for items in identity_history.values())
source_counts = Counter(
    identity.get("source", "direct")
    for frame_identities in identity_history.values()
    for identity in frame_identities.values()
)
resolved_person_ids = sorted({
    int(identity["person_id"])
    for frame_identities in identity_history.values()
    for identity in frame_identities.values()
})
summary = {
    "resolved_person_ids": resolved_person_ids,
    "person_count": len(resolved_person_ids),
    "assignment_count": assignment_count,
    "assignment_sources": dict(source_counts),
    "valid_face_samples": len(valid_face_samples),
    "persons_before_merge": initial_person_count,
    "persons_after_merge": merged_person_count,
    "tracks_with_face_identity": len(track_identity_anchors),
    "thresholds": {
        "min_face_quality": MIN_FACE_QUALITY,
        "cluster_eps": CLUSTER_EPS,
        "cluster_min_samples": CLUSTER_MIN_SAMPLES,
        "assignment_similarity": ASSIGNMENT_SIMILARITY,
        "merge_similarity": MERGE_SIMILARITY,
        "min_switch_jump_score": MIN_SWITCH_JUMP_SCORE,
        "min_fallback_overlap": MIN_FALLBACK_OVERLAP,
        "min_conflict_candidate_score": MIN_CONFLICT_CANDIDATE_SCORE,
    },
}

save_identity_resolution(
    RUN_DIRECTORY,
    identity_history=identity_history,
    switch_boundaries=switch_boundaries,
    summary=summary,
)
latest_path = update_latest_run(
    PROJECT_ROOT / "runs", RUN_DIRECTORY, stage="resolved"
)

print(f"Saved identities: {RUN_DIRECTORY / 'identities.jsonl'}")
print(f"Saved events: {RUN_DIRECTORY / 'identity_events.json'}")
print(f"Saved summary: {RUN_DIRECTORY / 'identity_summary.json'}")
print(f"Updated latest resolved run: {latest_path}")